In [1]:
import pandas as pd
import re
from pathlib import Path
import requests
import time
import shutil

try:
    import kagglehub
except ImportError as exc:
    raise ImportError(
        "kagglehub is required to download the raw ratings files. "
        "Install it with: pip install kagglehub"
    ) from exc

# Define project paths
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = "ramazanturann/user-animelist-dataset"
RAW_ANIMES_FILE = RAW_DIR / "kaggle_anime_source.csv"
RAW_RATINGS_FILE = RAW_DIR / "kaggle_user_ratings_source.csv"
LEGACY_RAW_ANIMES_FILE = RAW_DIR / "animes.csv"
LEGACY_RAW_RATINGS_FILE = RAW_DIR / "ratings.csv"
RATINGS_PROCESSED_FILE = PROCESSED_DIR / "ratings_processed.csv"

print("Working directory:", BASE_DIR)


Working directory: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect


c:\Users\CHAMPUX\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================================================
# ENSURE RAW KAGGLE FILES EXIST
# =========================================================
# Raw Kaggle CSVs are disposable. If they are missing, this cell downloads
# the KaggleHub dataset and copies the required source files into data/raw/.


def csv_has_columns(path, required_columns):
    try:
        columns = pd.read_csv(path, nrows=0).columns.str.strip()
    except Exception:
        return False

    return set(required_columns).issubset(set(columns))


def find_csv_with_columns(dataset_dir, required_columns):
    dataset_dir = Path(dataset_dir)

    for candidate in dataset_dir.rglob("*.csv"):
        if csv_has_columns(candidate, required_columns):
            return candidate

    return None


def move_legacy_raw_file_if_needed(legacy_path, target_path):
    if target_path.exists() or not legacy_path.exists():
        return

    legacy_path.replace(target_path)
    print(f"Migrated {legacy_path} -> {target_path}")


def copy_if_needed(source, target):
    source = Path(source)
    target = Path(target)

    if source.resolve() == target.resolve():
        return

    shutil.copy2(source, target)


def ensure_raw_kaggle_files():
    move_legacy_raw_file_if_needed(LEGACY_RAW_ANIMES_FILE, RAW_ANIMES_FILE)
    move_legacy_raw_file_if_needed(LEGACY_RAW_RATINGS_FILE, RAW_RATINGS_FILE)

    missing_files = [
        target.name
        for target in [RAW_ANIMES_FILE, RAW_RATINGS_FILE]
        if not target.exists()
    ]

    if not missing_files:
        print("Raw Kaggle files already exist in data/raw.")
        return

    print("Missing raw files:", missing_files)
    print("Downloading KaggleHub dataset:", KAGGLE_DATASET)

    dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print("Downloaded dataset path:", dataset_path)

    animes_source = find_csv_with_columns(
        dataset_path,
        required_columns={"animeID", "mal_url"}
    )

    ratings_source = find_csv_with_columns(
        dataset_path,
        required_columns={"userID", "animeID"}
    )

    if animes_source is None:
        raise FileNotFoundError(
            "Could not find anime metadata CSV with columns animeID and mal_url."
        )

    if ratings_source is None:
        raise FileNotFoundError(
            "Could not find ratings CSV with columns userID and animeID."
        )

    copy_if_needed(animes_source, RAW_ANIMES_FILE)
    copy_if_needed(ratings_source, RAW_RATINGS_FILE)

    print("Restored:", RAW_ANIMES_FILE)
    print("Restored:", RAW_RATINGS_FILE)


ensure_raw_kaggle_files()

# Load anime source metadata only. Ratings are processed in chunks later.
df = pd.read_csv(RAW_ANIMES_FILE)
df2_path = RAW_RATINGS_FILE

print("Dataset 1 shape:", df.shape)
print("Dataset 2 (ratings): LARGE FILE")


Missing raw files: ['animes.csv', 'ratings.csv']


100%|██████████| 1.91G/1.91G [01:10<00:00, 29.0MB/s]

Extracting files...


Downloaded dataset path: C:\Users\CHAMPUX\.cache\kagglehub\datasets\ramazanturann\user-animelist-dataset\versions\8
Restored: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\animes.csv
Restored: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\ratings.csv
Dataset 1 shape: (20237, 12)
Dataset 2 (ratings): LARGE FILE


In [3]:
# Strip whitespace from column names
df.columns = df.columns.str.strip()

In [4]:
# Extract MAL ID from URL
def extract_mal_id(url):
    if pd.isna(url):
        return None
    match = re.search(r"/anime/(\d+)", str(url))
    return int(match.group(1)) if match else None

df["mal_id"] = df["mal_url"].apply(extract_mal_id)

# Check results
print(df[["animeID", "mal_url", "mal_id"]].head())

   animeID                              mal_url  mal_id
0        1    https://myanimelist.net/anime/431     431
1        2   https://myanimelist.net/anime/1535    1535
2        3  https://myanimelist.net/anime/15315   15315
3        4  https://myanimelist.net/anime/14345   14345
4        5  https://myanimelist.net/anime/11757   11757


In [5]:
print("Total rows:", len(df))
print("Valid MAL IDs:", df["mal_id"].notna().sum())
print("Missing MAL IDs:", df["mal_id"].isna().sum())

Total rows: 20237
Valid MAL IDs: 20237
Missing MAL IDs: 0


In [6]:
# Map old animeID to new mal_id
df.rename(columns={"animeID": "old_animeID"}, inplace=True)
df.rename(columns={"mal_id": "animeID"}, inplace=True)

# Create mapping in memory. The map is single-use and is not saved.
id_map = df[["old_animeID", "animeID"]].dropna()
id_dict = dict(zip(id_map["old_animeID"], id_map["animeID"]))


In [7]:
# Process ratings in chunks (so RAM is not overwhelmed)
chunksize = 2_000_000
output_file = RATINGS_PROCESSED_FILE

first_chunk = True

for chunk in pd.read_csv(df2_path, chunksize=chunksize):
    # Map Kaggle anime IDs to MAL IDs
    chunk["animeID"] = chunk["animeID"].map(id_dict)

    # Filter valid rows without keeping a full copy in memory
    chunk = chunk[chunk["animeID"].notna()]

    # Optimize memory
    chunk["animeID"] = chunk["animeID"].astype("int32")
    chunk["userID"] = chunk["userID"].astype("int32")

    # Save incrementally
    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )

    first_chunk = False

print("Ratings processed and saved:", output_file)

# Raw source files are disposable after ratings_processed.csv exists.
if output_file.exists():
    for disposable_file in [RAW_ANIMES_FILE, RAW_RATINGS_FILE]:
        if disposable_file.exists():
            disposable_file.unlink()
            print("Deleted disposable raw file:", disposable_file)


Ratings processed and saved.
